# Event-Locked Temporal Factor Analysis

**What this notebook does.** Takes the TCA factor output of a completed face-rhythm run
and aligns the temporal factors to user-provided event markers (e.g. reward frames,
stimulus onsets), so you can see how facial rhythm factors respond around each event.

**Pipeline.** Load `analysis_files/*.h5` from a prior face-rhythm run -> window temporal
factors around each event -> plot trial-averaged factor traces (mean +/- std, heatmap) ->
visualize spatial factors with `Cmap_conjunctive` (magnitude x phase) -> optional
tiled trial-by-trial video playback.

**Inputs.** A face-rhythm project directory (see `demo_pipeline.ipynb`) plus a sequence
of per-trial event frame indices (one integer per trial).

**Outputs.** Figures (trial-averaged traces, heatmap, spatial-factor dot plots) saved
via `fr.util.Figure_Saver`; optionally a tiled trial-playback AVI.


In [ ]:
# ALWAYS RUN THIS CELL
# widen jupyter notebook window
from IPython.display import display, HTML
display(HTML("<style>.container {width:95% !important; }</style>"))

%load_ext autoreload
%autoreload 2
import face_rhythm as fr

from pprint import pprint
from pathlib import Path

import cv2

import numpy as np
import torch
import matplotlib.pyplot

fr.util.system_info(verbose=True);

## Enable interactive matplotlib (pan/zoom) for the figures below.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib notebook

In [ ]:
# ======== USER CONFIG ========
from pathlib import Path

# Face-rhythm project directory produced by demo_pipeline.ipynb
# (contains config.yaml, run_info.json, and analysis_files/).
DIR_PROJECT = Path(
    "/path/to/your/data"
)

# Directory holding the raw videos analyzed by the project.
DIR_VIDEOS = Path(
    "/path/to/your/data"
)

# Path where the tiled trial-playback video will be saved (optional section).
PATH_TILED_VIDEO_OUT = DIR_PROJECT / "visualizations" / "prep_factor_video.avi"

# Regex to filter videos in DIR_VIDEOS (Python `re` syntax).
FILENAME_REGEX = r".*\.avi$"

# Convert to strings for face_rhythm APIs that expect str paths.
directory_project = str(DIR_PROJECT)
directory_videos = str(DIR_VIDEOS)
filename_strMatch = FILENAME_REGEX

path_config, path_run_info, directory_project = fr.project.prepare_project(
    directory_project=directory_project,
    overwrite_config=False,  ## WARNING! Set True only if you want to wipe an existing config.
    update_project_paths=True,  ## Rewrite paths in config.yaml to match DIR_PROJECT on this machine.
    mkdir=True,
    initialize_visualization=False,  ## Set True only with a GUI cv2 build; headless cv2 crashes here.
    verbose=2,
)
figure_saver = fr.util.Figure_Saver(
    path_config=path_config,
    formats_save=['png'],
    kwargs_savefig={'bbox_inches': 'tight', 'pad_inches': 0.1, 'transparent': True, 'dpi': 300},
    overwrite=False,
    verbose=2,
)
# =============================


## Load face-rhythm analysis outputs

## Load project config and run info.

In [ ]:
config = fr.util.load_config_file(path_config)
run_info = fr.util.load_run_info_file(path_run_info)

## Load the four analysis HDF5 files: videos metadata, tracked points, VQT spectra, TCA factors.

In [ ]:
data_DatasetVideos = fr.h5_handling.simple_load(str(Path(directory_project) / 'analysis_files' / 'Dataset_videos.h5'))
pt_data = fr.h5_handling.simple_load(str(Path(directory_project) / 'analysis_files' / 'PointTracker.h5'))
spec_data = fr.h5_handling.simple_load(str(Path(directory_project) / 'analysis_files' / 'VQT_Analyzer.h5'))
tca_data = fr.h5_handling.simple_load(str(Path(directory_project) / 'analysis_files' / 'TCA.h5'))

## Pull out commonly-used fields: downsample factor, spectrogram time axis, example image, factors.

In [ ]:
ds_factor = config['VQT_Analyzer']['params_VQT']['downsample_factor']
spec_x_axis = [x for x in spec_data['x_axis'].values()]
image_example = data_DatasetVideos['example_image']
factors = tca_data['factors_rearranged']['0']

## Peek at the factor names available in the TCA output.

In [ ]:
print(f"Factor names: {factors.keys()}")

## Select and normalize the temporal factor(s)

The name of the temporal-factor axis inside the TCA `factors_rearranged[...]` dict depends on how `demo_pipeline.ipynb` called `tca.rearrange_data` (specifically `names_dims_array`). For the default pipeline that uses `names_dims_array=['xy', 'points', 'frequency', 'time']`, the temporal factor axis is named `'time'`. If your pipeline used the older default (`names_dims_array=[..., 'trials']`) set `name_tempFactor` to `'trials'` instead.

In [ ]:
name_tempFactor = 'time'  ## temporal-factor axis name; see the markdown above.
factors_temporal = factors[name_tempFactor]
if isinstance(factors_temporal, dict):
    factors_temporal = [f for f in factors_temporal.values()]
else:
    ## `method_handling_dictElements='separate'` leaves one temporal factor
    ## ndarray per session in `factors_rearranged[session_index]`; wrap in a
    ## list so downstream per-session iteration works uniformly.
    factors_temporal = [factors_temporal]

norm_factor = np.abs(np.concatenate(factors_temporal, axis=0)).std(0)


In [ ]:
# NOTE: Replace `event_frames` with your own per-trial event frame indices
# (e.g. reward frames or stimulus onsets), one integer per trial in
# `factors_temporal`. Use NaN for trials to skip. When your events live in a
# CSV/Excel file you can do:
#     import pandas as pd
#     event_frames = pd.read_csv(path_csv)["reward_frame"].to_numpy()
# For this demo we provide a hardcoded sequence of event frame indices spread
# across the session so the windowing logic below has trials to act on.
event_frames = [
    1000, 2500, 4000, 5500, 7000,
    8500, 10000, 11500, 13000, 14500,
    16000, 17500, 19000, 20500, 22000,
    23500, 25000, 26500, 28000, 29500,
]
event_times = np.asarray(event_frames, dtype=np.float32)


## Window each trial's temporal factor around its event frame.

In [ ]:
idx_reward = np.asarray(event_times, dtype=np.float32)
traces_windowed = []
windows = []
included = np.zeros((len(factors_temporal)), dtype=bool)
## Half-width is expressed in downsampled (VQT) time bins, giving a 60-bin window
## centered on each event. The `> 800` guard drops events too close to the start
## of the recording (before enough pre-event bins exist for a full window).
for ii,(trace, xaxis) in enumerate(zip(factors_temporal, spec_x_axis)):
    trace = np.abs(trace) / norm_factor[None,:]
    if (np.isnan(idx_reward[ii]) == False) and (idx_reward[ii] > 800):
        idx_reward_ds = np.argmin(np.abs(xaxis - idx_reward[ii]))
        window = (idx_reward_ds - 30, idx_reward_ds + 30)
        trace_windowed = trace[window[0]:window[1],:]
        traces_windowed.append(trace_windowed)
        windows.append(window)
        included[ii] = True
traces_windowed = np.stack(traces_windowed, axis=0)


## Plot frequency-axis factor loadings (log-x).

In [ ]:
plt.figure()
plt.plot(spec_data['frequencies'], tca_data['factors_rearranged']['0']['frequency'] / tca_data['factors_rearranged']['0']['frequency'].std(0));
plt.xscale('log')
plt.xlabel('frequency (Hz)')
plt.ylabel('factor amplitude (std units)')

## Plot trial-averaged temporal factor traces (mean line + std shading, event marker).

In [ ]:
plt.figure()
for ii,trace in enumerate(traces_windowed.transpose(2,0,1)[:]):
    plt.plot(np.arange(trace.shape[1])/20, trace.mean(0))
for ii,trace in enumerate(traces_windowed.transpose(2,0,1)[:]):
    plt.fill_between(np.arange(trace.shape[1])/20, trace.mean(0) + trace.std(0), trace.mean(0) - trace.std(0), alpha=0.3)

plt.plot([2,2], [0,5], 'k')
plt.xlabel('delta time (s)')
plt.ylabel('factor amplitude (std units)')
plt.legend(np.arange(traces_windowed.shape[2])+1)


## Heatmap of trial-averaged factor amplitudes across time.

In [ ]:
plt.figure()
plt.imshow(traces_windowed.mean(0).T, aspect='auto')
plt.xlabel('time bin (downsampled)')
plt.ylabel('factor index')
plt.colorbar()


## Build complex-valued (x + iy) spatial factors from the paired xy-loadings, then split into magnitude and angle.

In [ ]:
frame_visualizer = fr.visualization.FrameVisualizer(frame_height_width=image_example.shape[:2])

factors_dots_complex = np.stack([p[:p.shape[0]//2,] + 1j*p[p.shape[0]//2:] for p in tca_data['factors_rearranged']['0']['(xy points)'].T], axis=1)

factors_dots_mag = np.abs(factors_dots_complex)
factors_dots_angle = np.angle(factors_dots_complex)

## Build a conjunctive colormap: lightness encodes magnitude, hue encodes phase angle.

In [ ]:
cmap_mag   = fr.helpers.simple_cmap(colors=[[0,0,0],[1,1,1]], name='mag')
cmap_angle = fr.helpers.simple_cmap(colors=[[0,0,1],[0.7,0,0.7],[1,0,0]], name='angle')
cmap_conj = fr.helpers.Cmap_conjunctive([cmap_mag, cmap_angle], normalize=True)

## Plot each spatial factor as colored dots overlaid on the example face frame.

In [ ]:
n_factors = factors_dots_mag.shape[1]
fig, axs = plt.subplots(n_factors, 1, figsize=(6, 40))
for ii, (dots_mag, dots_angle) in enumerate(zip(factors_dots_mag.T, factors_dots_angle.T)):
    frame_with_points = frame_visualizer.visualize_image_with_points(
        image=image_example,
        points=[pt_data['point_positions'].astype(np.int64) for _ in range(n_factors)],
        point_sizes=6,
        points_colors=[(cmap_conj(np.stack((dots_mag, dots_angle), axis=1))[:,:3])],
        alpha=0.6,
    )
    axs[ii].imshow(frame_with_points)
    axs[ii].axis('off')


## Tiled per-trial video preview

Stitch a grid of event-windowed video clips into one AVI for visual QC of trial alignment.

In [ ]:
## Re-resolve video paths against DIR_VIDEOS on this machine, in case run_info.json
## was written from a different host (paths_videos in the run_info can be stale).
paths_videos_resolved = fr.helpers.find_paths(
    dir_outer=directory_videos,
    reMatch=filename_strMatch,
    depth=0,
)
assert len(paths_videos_resolved) >= len(factors_temporal), (
    f"FR ERROR: found {len(paths_videos_resolved)} videos under {directory_videos} but "
    f"the project has {len(factors_temporal)} trials — re-run the pipeline or adjust paths."
)
paths_videos = np.array(paths_videos_resolved[:len(factors_temporal)])[included]

## Convert downsampled-frame windows back to raw video-frame indices.
windows_raw = (np.asarray(windows) * int(ds_factor)).astype(int)


In [ ]:
## Tile per-trial video windows into a single grid video for preview.
## Per-trial video clips are read into memory as ndarrays and tiled by
## ``fr.helpers.make_tiled_video_array``.

def read_video_frames_window(path_video, frame_start, frame_stop):
    """Read frames [frame_start, frame_stop) from path_video into an ndarray."""
    cap = cv2.VideoCapture(str(path_video))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_start))
    frames = []
    for _ in range(int(frame_stop) - int(frame_start)):
        ok, frame = cap.read()
        if not ok:
            break
        ## cv2 reads BGR; convert to RGB for consistency with make_tiled_video_array.
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames, axis=0).astype(np.uint8) if frames else np.zeros((0, 0, 0, 3), dtype=np.uint8)


## Read per-trial clips (take up to 36 for a 6x6 tile).
n_clips = min(36, len(paths_videos))
videos_clips = []
for ii in range(n_clips):
    path_vid = str(paths_videos[ii])
    fstart, fstop = int(windows_raw[ii, 0]), int(windows_raw[ii, 1])
    clip = read_video_frames_window(path_vid, fstart, fstop)
    if clip.shape[0] == 0:
        continue
    videos_clips.append(clip)

video_tiled = fr.helpers.make_tiled_video_array(
    videos=videos_clips,
    shape=None,
    verbose=True,
)


## Save the tiled preview AVI (set `show=True` for live playback).

In [ ]:
if len(video_tiled) > 0:
    PATH_TILED_VIDEO_OUT.parent.mkdir(parents=True, exist_ok=True)
    fr.helpers.play_video_cv2(video_tiled, frameRate=240, path_save=str(PATH_TILED_VIDEO_OUT), show=False)
else:
    print("Skipping tiled-video save: no clips were read.")
